## 연습 1: ClearCover Insurance 데이터 플랫폼 설정

데이터 품질 파이프라인을 실행하려면 먼저 Unity Catalog 구조와 원시 소스 데이터가 준비되어 있어야 합니다.

이 노트북의 각 셀을 **위에서 아래로** 실행하세요. 설정이 완료되면 실습 안내 페이지로 돌아가 연습 2를 계속 진행하세요.

### 작업 1: Unity Catalog 구조 만들기

이 셀은 미리 제공된 것입니다. 환경을 준비하려면 실행하세요.

다음 Unity Catalog 객체를 생성합니다:
- `insurance_lab`라는 이름의 **catalog** — ClearCover Insurance 플랫폼의 최상위 네임스페이스
- `insurance_lab` 내부의 **bronze schema** — 소스 시스템에서 도착한 그대로의 원시 청구 데이터
- `insurance_lab` 내부의 **silver schema** — 검증되고 형식이 안전한 청구 데이터
- `insurance_lab` 내부의 **gold schema** — 집계된 보고용 데이터
- `insurance_lab.bronze` 내부의 `raw_files`라는 이름의 **volume** — 들어오는 CSV 청구 파일의 랜딩 존

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_ws*")]
dplab09_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dplab09_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS insurance_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'ClearCover Insurance 분석 플랫폼'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS insurance_lab.bronze
  COMMENT '소스 시스템에서 받은 그대로 수정되지 않은 원시 청구 데이터';

CREATE SCHEMA IF NOT EXISTS insurance_lab.silver
  COMMENT '검증된 청구 데이터, 형식 검사 및 품질 제약이 적용됨';

CREATE SCHEMA IF NOT EXISTS insurance_lab.gold
  COMMENT '대시보드용 집계 보고 데이터';

CREATE VOLUME IF NOT EXISTS insurance_lab.bronze.raw_files
  COMMENT '원시 CSV 청구 파일을 위한 랜딩 존';

### 작업 2: 원시 청구 CSV를 볼륨에 기록하기

이 데이터셋에는 **의도적인 품질 문제**가 포함되어 있습니다: 누락된 식별자, 잘못된 형식의 날짜, 유효하지 않은 금액, 잘못된 상태 값 — 파이프라인이 해결할 바로 그 문제들입니다.

> `claim_date`와 `claim_amount`는 의도적으로 **STRING** 컬럼으로 로드됩니다. 형식 검증은 파이프라인의 역할입니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 원시 청구 CSV를 볼륨에 기록합니다
volume_path = '/Volumes/insurance_lab/bronze/raw_files'

rows = [
    'claim_id,policy_id,customer_id,claim_date,incident_date,claim_amount,claim_type,status,coverage_amount,agent_code',
    'CLM-001,POL-1001,CUST-001,2026-01-05,2026-01-03,4500.00,AUTO,OPEN,25000.00,AGT-01',
    'CLM-002,POL-1002,CUST-002,2026-01-07,2026-01-06,12000.00,HOME,PENDING,150000.00,AGT-02',
    'CLM-003,POL-1003,CUST-003,2026-01-10,2026-01-09,250.00,HEALTH,CLOSED,5000.00,AGT-01',
    'CLM-004,,CUST-004,2026-01-12,2026-01-11,8900.00,AUTO,OPEN,30000.00,AGT-03',
    'CLM-005,POL-1005,CUST-005,2026-01-14,2026-01-13,,LIFE,PENDING,200000.00,AGT-02',
    ',POL-1006,CUST-006,2026-01-15,2026-01-14,3200.00,HOME,CLOSED,80000.00,AGT-04',
    'CLM-007,POL-1007,CUST-007,not-a-date,2026-01-16,7600.00,AUTO,OPEN,35000.00,AGT-03',
    'CLM-008,POL-1008,CUST-008,2026-01-18,2026-01-17,N/A,HEALTH,PENDING,10000.00,AGT-01',
    'CLM-009,POL-1009,CUST-009,2026-01-20,2026-01-19,-500.00,HOME,INVALID_STATUS,120000.00,AGT-04',
    'CLM-010,POL-1010,CUST-010,2026-01-22,2026-01-21,15000.00,VEHICLE,CLOSED,50000.00,AGT-02',
    'CLM-011,POL-1011,CUST-011,2026-01-25,2026-01-24,1800.00,AUTO,PENDING,20000.00,AGT-05',
    'CLM-012,POL-1012,CUST-012,2026-01-27,2026-01-26,6400.00,HEALTH,OPEN,15000.00,AGT-01',
    'CLM-013,POL-1013,CUST-013,2026-01-28,2026-01-27,92000.00,LIFE,CLOSED,300000.00,AGT-03',
    'CLM-014,POL-1014,CUST-014,2026-01-30,invalid-date,-900.00,AUTO,PENDING,22000.00,AGT-02',
    'CLM-015,POL-1015,CUST-015,2026-02-01,2026-01-31,3750.00,HOME,OPEN,95000.00,AGT-05',
    'CLM-016,POL-1016,CUST-016,2026-02-03,2026-02-02,550.00,HEALTH,CLOSED,8000.00,AGT-02',
    'CLM-017,POL-1017,CUST-017,2026-02-05,2026-02-04,18500.00,LIFE,PENDING,250000.00,AGT-03',
    'CLM-018,POL-1018,CUST-018,2026-02-07,2026-02-06,2200.00,AUTO,OPEN,28000.00,AGT-01',
    'CLM-019,POL-1019,,2026-02-09,2026-02-08,4100.00,HOME,PENDING,110000.00,AGT-04',
    'CLM-020,POL-1020,CUST-020,2026-02-11,2026-02-10,67000.00,LIFE,CLOSED,400000.00,AGT-03'
]

dbutils.fs.put(volume_path + '/insurance_claims_raw.csv', '\n'.join(rows) + '\n', overwrite=True)
print('✅ CSV가 기록된 위치:', volume_path)

### 작업 3: CSV를 bronze Delta 테이블로 로드하기

명시적 스키마에 주목하세요 — `claim_date`와 `claim_amount`는 파이프라인의 형식 검사 연습이 올바르게 작동하도록 **STRING**으로 유지됩니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — CSV를 bronze Delta 테이블로 로드합니다
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField('claim_id',        StringType(), True),
    StructField('policy_id',       StringType(), True),
    StructField('customer_id',     StringType(), True),
    StructField('claim_date',      StringType(), True),   # 의도적으로 STRING
    StructField('incident_date',   StringType(), True),
    StructField('claim_amount',    StringType(), True),   # 의도적으로 STRING
    StructField('claim_type',      StringType(), True),
    StructField('status',          StringType(), True),
    StructField('coverage_amount', DoubleType(), True),
    StructField('agent_code',      StringType(), True),
])

volume_path = '/Volumes/insurance_lab/bronze/raw_files'

df = (
    spark.read
    .option('header', 'true')
    .option('nullValue', '')
    .schema(schema)
    .csv(volume_path + '/insurance_claims_raw.csv')
)

(
    df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('insurance_lab.bronze.claims_raw')
)

print('✅ 테이블 insurance_lab.bronze.claims_raw 생성됨')
print(f'   로드된 총 행 수: {df.count()}')

### 검증: 원시 데이터 살펴보기

아래 셀을 실행하여 테이블이 성공적으로 생성되었는지 확인하고 데이터를 미리 봅니다.

품질 문제가 있는 행을 결과에서 찾아보세요 — 이러한 행들이 파이프라인 연습의 대상입니다.

In [ ]:
%sql
SELECT *
FROM insurance_lab.bronze.claims_raw
ORDER BY claim_id NULLS LAST;

In [ ]:
%sql
DESCRIBE TABLE insurance_lab.bronze.claims_raw;

---

✅ **설정 완료!** 실습 안내 페이지로 돌아가 연습 2를 계속 진행하세요.

In [ ]:
%sql
-- 선택적 정리 — 완료되면 주석을 해제하여 모든 실습 리소스를 제거하세요
-- DROP CATALOG IF EXISTS insurance_lab CASCADE;